# Phase 1 of 3: Data Generation (v4)

**Run order:** 1 of 4 notebooks (`phase_1` → `phase_2` → `phase_3` → `phase_4_eval`).
**Needs:** network access to Hugging Face on the first run (later runs read `data/source-raw/`).
**Produces:** `data/source-clean-v4/{train,val,test,adversarial_test,lora_holdout}.parquet` and `manifest.json`. v1 to v3 stay frozen in their own folders.
**Expect:** the first run downloads the new sources (LLMail is the bulk, about 300 MB of JSON); MinHash over roughly 110k rows takes several minutes on CPU; the last cell must print `leakage checks passed` and a counts line per output file.

**Why v4 exists (v3 verdict, 2026-10-09).** On v3 the CNN + BiGRU lost the promotion gate to the TF-IDF
baseline at every FPR budget (recall at FPR 1%: 0.577 vs 0.641). The product scans retrieved documents,
but 82% of the v3 test set was chat-shaped and the document-shaped slice held 53 injected rows from one
source (BIPIA). Four of nine reasons had zero training rows. v4 fixes the data, not the model:

1. **Document-shaped attacks from several sources, thousands of rows.** Real adaptive attack emails from
   the LLMail-Inject challenge, injected tasks planted in RAG passages (PIArena), instructions hidden in
   tool-returned records (Nemotron agentic IPI), and injections inside real Kubernetes manifests.
2. **Matched benign counterparts for every document source**: the clean passage, the clean record, the
   clean manifest, and the 203 clean emails LLMail used for its own false-positive tests, so document
   shape, email shape, JSON shape and "has a system-like banner" cannot stand in for "contains an attack".
3. **A third shape, `agentic`**: tool requests, agent messages, memory and transcripts (3nesdeniz agentic
   sets), each attack paired with a benign row from the same workflow that uses the same vocabulary.
4. **The four empty reasons get rows** through the attack-family maps below: Fake System Injection,
   Delimiter Confusion, Role-Play Bypass, Chain-of-Instructions.
5. **Harder negatives**: everyday instructions, shell commands and code (ogma), real non-jailbreak prompt
   templates from Reddit and Discord (in-the-wild regular), AI-safety questions (S-Labs benign), and the
   NotInject over-defense set, which is forced into `test` so it is graded, never trained on.
6. **LLMail rows are re-rendered** from `Subject of the email: X.   Body: Y` to the same `Subject: X\n\nY`
   layout the legitimate emails use, so the challenge wrapper cannot become the signal.
7. The 6-word template-twin rate is now reported for document-shaped positives too.

Everything from v3 is kept: BIPIA keyed by normalized base text and capped per document, held-out
isolation of the LoRA comparison sets, grouped balanced splits, train-only augmentation.

**Sources (verified on Hugging Face 2026-10-09; licences on the dataset cards):**

| Source | Licence | Rows used | Label | Reason | Shape |
|---|---|---|---|---|---|
| `nvidia/Nemotron-Safety-Guard-Dataset-v3`, English `jailbreaking` | CC-BY-4.0 | all | INJECTION | Jailbreak Technique | chat |
| `allenai/wildguardmix` `wildguardtrain` | ODC-BY | all | by harm label | LoRA subcategory map | chat |
| `deepset/prompt-injections` | Apache-2.0 | 662 | by label | Override Attempt | chat |
| `jackhhao/jailbreak-classification` | Apache-2.0 | 1,306 | by type | Jailbreak Technique | chat |
| `geodesic-research/bipia` | CC-BY-SA-4.0 | ≤3 attacks per base document, plus one clean copy | both | Embedded Instruction | document |
| `puyang2025/seven-phishing-email-datasets`, label 0 | per component corpus | ≤3,000 | SAFE | — | document |
| `wikimedia/wikipedia` `20231101.en` | CC-BY-SA-3.0 | ≤6,000 paragraphs | SAFE | — | document |
| **`microsoft/llmail-inject-challenge`** labelled unique submissions, `api_triggered` only | MIT | ≤8,000 sampled | INJECTION | Embedded Instruction | document |
| **`microsoft/llmail-inject-challenge`** `emails_for_fp_tests` | MIT | 203 | SAFE | — | document |
| **`sleeepeer/PIArena`** clean context and context with the injected task planted at a sentence boundary | MIT | ≤2,100 pairs | both | Embedded Instruction | document |
| **`nvidia/Nemotron-RL-Agentic-Indirect-Prompt-Injection-v1`** injected record field, plus the clean fields of the same name | CC-BY-4.0 | 1,272 + clean | both | Embedded Instruction | document |
| **`rgeada/k8s-resource-prompt-injection`** paired manifests, ≤3,000 chars | Apache-2.0 | ≤2,500 pairs | both | Embedded Instruction | document |
| **`3nesdeniz/agentic-prompt-injection-5k`** | CC-BY-4.0 | 4,935 | by label | family map | agentic |
| **`3nesdeniz/agentic-prompt-injection-boundary-pairs`** | CC-BY-4.0 | 1,200 | by label | family map | by source context |
| **`3nesdeniz/english-prompt-injection-3k`** | CC-BY-4.0 | 3,053 | by label | family map | chat |
| **`TrustAIRLab/in-the-wild-jailbreak-prompts`** `jailbreak_2023_12_25` / `regular_2023_12_25` | MIT | 1,405 / ≤5,000 | by config | Jailbreak Technique | chat |
| **`axiotic/ogma-prompt-injection-benign`** | Apache-2.0 | ≤6,000 | SAFE | — | chat |
| **`S-Labs/prompt-injection-dataset`**, benign rows only (positives carry no category, so no reason) | MIT | ≤3,000 | SAFE | — | chat |
| **`leolee99/NotInject`** | MIT | 339 | SAFE | — | chat, forced into `test` |

Rejected after verification (recorded in the manifest): non-commercial licences (`rogue-security/prompt-injections-benchmark`, `Mindgard/evaded-prompt-injection-and-jailbreak-samples`), gated (`hackaprompt/hackaprompt-dataset`, `Necent/llm-jailbreak-prompt-injection-dataset`, `prodnull/prompt-injection-repo-dataset`), no licence (`xTRam1/safe-guard-prompt-injection`, `jayavibhav/prompt-injection-safety`, `guychuk/open-prompt-injection`, `xxz224/prompt-injection-attack-dataset`, `hirundo-io/prompt-injection-purple-llama`), heuristic labels (`Smooth-3/llm-prompt-injection-attacks`).

`Horizon-Labs/prompt-injection-eval-suite` is an evaluation suite, not a training source; it contains the
test splits of the 3nesdeniz sets, so its `agentic5k_test` and `boundary_pairs_test` configs are contaminated
for any model trained on v4 and must be excluded if the suite is used.

**Held out, never trained on:** `wildguardtest` and the Nemotron test-split English `jailbreaking` rows, as in v1 to v3.

**Reason labels are weak.** Every reason is map-derived from a source, subcategory, or attack family, not
human annotated. All nine reasons have training rows after this build; Delimiter Confusion and
Chain-of-Instructions have the fewest (a few hundred). Report, do not gate.

**Safety note:** the data contains live attack strings. Never paste them into a real model session outside these notebooks.


## 1. Configuration


In [1]:
import hashlib, json, os
from pathlib import Path

SEED = 42
DATASET_VERSION = "v4"        # v1 to v3 outputs are frozen in their own folders, never overwritten

# Resolve the project root so this notebook runs unchanged on the DGX Spark (container or
# host), a Mac, or any other checkout: env var first, then the Spark paths, then walk up
# from the working directory until a folder containing docs/PLAN.md is found.
def resolve_project_dir():
    candidates = [
        Path(p) for p in (
            os.environ.get("PI_NN_DIR"),
            "/workspace/training/prompt-injection-nn",
            "/home/spark/projects/training/prompt-injection-nn",
        ) if p
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError(f"project root not found; tried {candidates} and parents of {Path.cwd()}")


PROJECT_DIR = resolve_project_dir()
RAW_DIR = PROJECT_DIR / "data" / "source-raw"
CLEAN_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face sources. Schemas, licenses and splits verified 2026-10-09; see the table above.
HF_NEMOTRON = "nvidia/Nemotron-Safety-Guard-Dataset-v3"
HF_WILDGUARD = "allenai/wildguardmix"           # configs: wildguardtrain (train), wildguardtest (test); gated, auto-approved
HF_DEEPSET = "deepset/prompt-injections"       # text, label (1 = injection); splits train/test
HF_JACKHHAO = "jackhhao/jailbreak-classification"  # prompt, type (jailbreak | benign); splits train/test
HF_BIPIA = "geodesic-research/bipia"           # context already contains attack_str at `position`
HF_WIKIPEDIA = "wikimedia/wikipedia"
HF_EMAIL = "puyang2025/seven-phishing-email-datasets"  # text, subject, label (0 = legitimate), dataset_name; shared with email-classifier-nn
WIKIPEDIA_CONFIG = "20231101.en"
# v4 sources (schemas verified by inspection 2026-10-09).
HF_LLMAIL = "microsoft/llmail-inject-challenge"                      # labelled unique submissions (dict text -> {attack_attempt, reason}); emails_for_fp_tests (list of 203 clean emails)
HF_PIARENA = "sleeepeer/PIArena"                                     # 16 task splits; context, injected_task, category
HF_NEMOTRON_IPI = "nvidia/Nemotron-RL-Agentic-Indirect-Prompt-Injection-v1"  # train.jsonl; injection.injection_text hidden in environment[...][injection_vector]
HF_K8S = "rgeada/k8s-resource-prompt-injection"                       # text, label (safe|injection), vector, manifest_id
HF_AGENTIC5K = "3nesdeniz/agentic-prompt-injection-5k"               # text, label, attack_family, technique
HF_BOUNDARY = "3nesdeniz/agentic-prompt-injection-boundary-pairs"    # text, label (str), attack_family, pair_id, scenario_id, source_context
HF_ENGLISH3K = "3nesdeniz/english-prompt-injection-3k"               # text, label, attack_family
HF_ITW = "TrustAIRLab/in-the-wild-jailbreak-prompts"                 # configs jailbreak_2023_12_25, regular_2023_12_25; prompt
HF_OGMA = "axiotic/ogma-prompt-injection-benign"                      # text, label 0, subtype
HF_SLABS = "S-Labs/prompt-injection-dataset"                          # text, label
HF_NOTINJECT = "leolee99/NotInject"                                   # prompt, word_list, category; three subsets
HF_DATASETS = (HF_NEMOTRON, f"{HF_WILDGUARD}/wildguardtrain", f"{HF_WILDGUARD}/wildguardtest",
               HF_DEEPSET, HF_JACKHHAO, HF_BIPIA, f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", HF_EMAIL,
               HF_LLMAIL, HF_PIARENA, HF_NEMOTRON_IPI, HF_K8S, HF_AGENTIC5K, HF_BOUNDARY, HF_ENGLISH3K,
               f"{HF_ITW}/jailbreak_2023_12_25", f"{HF_ITW}/regular_2023_12_25", HF_OGMA, HF_SLABS, HF_NOTINJECT)
EXCLUDED_SOURCES = {
    "xTRam1/safe-guard-prompt-injection": "no license on the dataset card",
    "rogue-security/prompt-injections-benchmark": "CC-BY-NC-4.0, non-commercial",
    "Mindgard/evaded-prompt-injection-and-jailbreak-samples": "CC-BY-NC-4.0, non-commercial",
    "hackaprompt/hackaprompt-dataset": "gated",
    "Necent/llm-jailbreak-prompt-injection-dataset": "gated",
    "prodnull/prompt-injection-repo-dataset": "gated",
    "jayavibhav/prompt-injection-safety": "no license",
    "guychuk/open-prompt-injection": "no license",
    "xxz224/prompt-injection-attack-dataset": "no license",
    "hirundo-io/prompt-injection-purple-llama": "no license, empty card",
    "Smooth-3/llm-prompt-injection-attacks": "labels are rule-derived from other sets; noisy",
    "Horizon-Labs/prompt-injection-eval-suite": "evaluation suite, not a training source; contains the 3nesdeniz test splits",
    "S-Labs/prompt-injection-dataset (positives)": "no attack category, so no reason label; benign rows are used",
}

# Caps. 0 means every eligible row. Applied with a seeded sample after filtering, before dedupe.
NEMOTRON_MAX = 0
WILDGUARD_INJECTION_MAX = 0
WILDGUARD_SAFE_MAX = 0
BIPIA_ROWS_PER_DOC = 3                # 182 base documents x up to 3 attacks each (v2: 48 per attack string, 40:1 injected:clean)
WIKIPEDIA_MAX_ARTICLES = 1500         # streamed in dataset order, deterministic
WIKIPEDIA_MAX_PARAGRAPHS = 6000
WIKIPEDIA_MIN_PARAGRAPH_CHARS = 200
EMAIL_HAM_MAX = 3000                  # legitimate emails as structured document-shaped SAFE rows
EMAIL_MIN_BODY_CHARS = 200
BIPIA_SPLIT_SEED = SEED + 7           # independent 80/10/10 partitions of BIPIA documents and attack strings
# v4 caps and limits.
LLMAIL_MAX = 8000                     # api_triggered attack emails sampled across both phases (24,559 available)
LLMAIL_MAX_CHARS = 6000
PIARENA_MAX_CONTEXT_CHARS = 1500      # clean passage cut at a sentence boundary so the planted task stays inside the 512-token window
K8S_MAX_PAIRS = 2500
K8S_MAX_CHARS = 3000
ITW_REGULAR_MAX = 5000
ITW_MAX_CHARS = 6000
OGMA_MAX = 6000
SLABS_BENIGN_MAX = 3000

MIN_TEXT_CHARS = 8                    # "Ignore the above" is a valid 16-char injection; keep the gate low
NEAR_DUP_THRESHOLD = 0.80             # MinHash LSH Jaccard threshold (v1: 0.90)
MINHASH_PERMS = 128
SHINGLE_SIZE = 5                      # word shingles
TEMPLATE_KEY_WORDS = 8                # chat-shaped positives sharing this many leading words share a split (v1: 12)
SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
AUG_FRACTION = 0.25                   # fraction of train positives that get an obfuscated copy

# Output contract shared with the LoRA and the Open WebUI filter: "SAFE" or "INJECTION: <reason>".
REASONS = [
    "Override Attempt",
    "Fake System Injection",
    "Meta-Command Injection",
    "Jailbreak Technique",
    "Delimiter Confusion",
    "Privilege Escalation",
    "Embedded Instruction",
    "Role-Play Bypass",
    "Chain-of-Instructions",
]

# Copied from safety/notebooks/prompt_injection_lora_qwen38_27b.ipynb. WildGuardMix subcategories
# not listed here map to Jailbreak Technique, as in the LoRA.
SUBCATEGORY_REASON_MAP = {
    "cyberattack": "Meta-Command Injection",
    "fraud_assisting_illegal_activities": "Jailbreak Technique",
    "sensitive_information_organization_government": "Privilege Escalation",
    "private_information_individual": "Privilege Escalation",
    "copyright_violations": "Jailbreak Technique",
    "defamation_encouraging_unethical_or_unsafe_actions": "Jailbreak Technique",
    "others": "Jailbreak Technique",
}
DEFAULT_REASON = "Jailbreak Technique"

# v4 attack-family maps onto the nine reasons. Family names are the datasets' own.
AGENTIC5K_FAMILY_REASON = {
    "rag_document_embedded": "Embedded Instruction",
    "tool_function_abuse": "Meta-Command Injection",
    "approval_authority_escalation": "Privilege Escalation",
    "memory_trust_boundary": "Fake System Injection",
}
AGENTIC5K_TECHNIQUE_REASON = {"delimiter_confusion": "Delimiter Confusion"}   # technique overrides family where it names a reason
ENGLISH3K_FAMILY_REASON = {
    "instruction_override": "Override Attempt",
    "system_prompt_extraction": "Privilege Escalation",
    "jailbreak_persona": "Role-Play Bypass",
    "delimiter_format_injection": "Delimiter Confusion",
    "obfuscation_encoding": "Jailbreak Technique",
    "data_exfiltration": "Privilege Escalation",
    "refusal_suppression": "Jailbreak Technique",
    "payload_splitting": "Chain-of-Instructions",
}
BOUNDARY_FAMILY_REASON = {
    "direct_instruction_override": "Override Attempt",
    "system_prompt_extraction": "Privilege Escalation",
    "roleplay_jailbreak": "Role-Play Bypass",
    "authority_claim_bypass": "Privilege Escalation",
    "sensitive_data_exfiltration": "Privilege Escalation",
    "tool_action_abuse": "Meta-Command Injection",
    "indirect_content_injection": "Embedded Instruction",
    "rag_context_poisoning": "Embedded Instruction",
    "memory_context_poisoning": "Fake System Injection",
    "obfuscation_encoding": "Jailbreak Technique",
    "multi_agent_trust_spoofing": "Fake System Injection",
    "approval_workflow_bypass": "Privilege Escalation",
}
BOUNDARY_CONTEXT_SHAPE = {
    "direct_user": "chat", "chat_message": "chat",
    "agent_tool_request": "agentic", "agent_message": "agentic", "conversation_memory": "agentic", "tool_output": "agentic",
}  # every other source_context (retrieved_document, email, pdf_document, code_comment, ...) is "document"

CONFIG_FINGERPRINT = hashlib.sha256(json.dumps({
    "version": DATASET_VERSION, "datasets": HF_DATASETS, "excluded": EXCLUDED_SOURCES,
    "caps": {"nemotron": NEMOTRON_MAX, "wg_inj": WILDGUARD_INJECTION_MAX, "wg_safe": WILDGUARD_SAFE_MAX,
             "bipia_per_doc": BIPIA_ROWS_PER_DOC, "wiki_articles": WIKIPEDIA_MAX_ARTICLES,
             "wiki_paragraphs": WIKIPEDIA_MAX_PARAGRAPHS, "wiki_min_chars": WIKIPEDIA_MIN_PARAGRAPH_CHARS,
             "email_ham": EMAIL_HAM_MAX, "email_min_chars": EMAIL_MIN_BODY_CHARS, "bipia_split_seed": BIPIA_SPLIT_SEED,
             "llmail": LLMAIL_MAX, "llmail_chars": LLMAIL_MAX_CHARS, "piarena_chars": PIARENA_MAX_CONTEXT_CHARS,
             "k8s_pairs": K8S_MAX_PAIRS, "k8s_chars": K8S_MAX_CHARS, "itw_regular": ITW_REGULAR_MAX, "itw_chars": ITW_MAX_CHARS,
             "ogma": OGMA_MAX, "slabs_benign": SLABS_BENIGN_MAX},
    "family_maps": {"agentic5k": AGENTIC5K_FAMILY_REASON, "agentic5k_technique": AGENTIC5K_TECHNIQUE_REASON,
                    "english3k": ENGLISH3K_FAMILY_REASON, "boundary": BOUNDARY_FAMILY_REASON, "boundary_shape": BOUNDARY_CONTEXT_SHAPE},
    "min_chars": MIN_TEXT_CHARS, "near_dup": NEAR_DUP_THRESHOLD, "perms": MINHASH_PERMS,
    "shingle": SHINGLE_SIZE, "template_words": TEMPLATE_KEY_WORDS, "splits": SPLITS,
    "aug": AUG_FRACTION, "seed": SEED, "reasons": REASONS, "subcategory_map": SUBCATEGORY_REASON_MAP,
    "split_alloc": "balanced-rows-label-source-reason-shape; bipia joint doc+template partition, normalized doc key; v4 paired sources share a group key; notinject forced to test; boundary scenarios forced seeded partition, excluded from LSH and template merge",
}, sort_keys=True).encode()).hexdigest()
print("dataset version:", DATASET_VERSION)
print("output dir:", CLEAN_DIR)
print("config fingerprint:", CONFIG_FINGERPRINT[:16])


dataset version: v4
output dir: /workspace/training/prompt-injection-nn/data/source-clean-v4
config fingerprint: 7f42d9d83806880d


## 2. Environment


In [2]:
%pip install -q datasets pandas pyarrow datasketch

import gc, re, unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH

WS_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[a-z0-9]+")


def normalize_for_fp(text):
    return WS_RE.sub(" ", (text or "").lower()).strip()


def fingerprint(text):
    return hashlib.sha256(normalize_for_fp(text).encode("utf-8", "replace")).hexdigest()


def template_key(text, words=TEMPLATE_KEY_WORDS):
    # Leading words of the normalized text. Jailbreak corpora are generated from a few dozen
    # skeletons ("From now on you are going to act as ..."); rows sharing a skeleton share a split.
    tokens = WORD_RE.findall(normalize_for_fp(text))
    return " ".join(tokens[:words]) if len(tokens) >= words else None


def short_hash(text):
    return hashlib.sha256(text.encode("utf-8", "replace")).hexdigest()[:16]


Note: you may need to restart the kernel to use updated packages.


## 3. Download Hugging Face Sources

Every source is loaded with `datasets.load_dataset` and cached as parquet in `data/source-raw/`;
re-runs read the local cache. Only the rows this build uses are cached (for Nemotron, the English
`jailbreaking` rows of the train and test splits, not the 450k-row corpus). Wikipedia is streamed
in dataset order and cut off after `WIKIPEDIA_MAX_ARTICLES`, which is deterministic.


In [3]:
def load_hf_cache(name, cache_path, build):
    if cache_path.exists():
        frame = pd.read_parquet(cache_path)
        print(f"{name} (cached): {len(frame):,} rows")
        return frame
    frame = build()
    tmp = Path(str(cache_path) + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, cache_path)
    gc.collect()
    print(f"{name} (downloaded): {len(frame):,} rows")
    return frame


def nemotron_jailbreaking(split):
    def build():
        dataset = load_dataset(HF_NEMOTRON, split=split)
        kept = dataset.filter(
            lambda x: x.get("tag") == "jailbreaking"
            and x.get("language") in (None, "en")
            and x.get("prompt") not in (None, "REDACTED")
        )
        return kept.to_pandas()[["id", "prompt", "prompt_label"]]
    return load_hf_cache(f"{HF_NEMOTRON}:{split}:jailbreaking:en", RAW_DIR / f"nemotron_{split}_jailbreaking_en.parquet", build)


def wildguard(config, split):
    cols = ["prompt", "adversarial", "prompt_harm_label", "subcategory"]
    return load_hf_cache(f"{HF_WILDGUARD}/{config}", RAW_DIR / f"wildguard_{config}.parquet",
                         lambda: load_dataset(HF_WILDGUARD, config, split=split).to_pandas()[cols])


def with_splits(dataset_id, cache_name):
    def build():
        dd = load_dataset(dataset_id)
        parts = []
        for split_name, part in dd.items():
            frame = part.to_pandas()
            frame["hf_split"] = split_name
            parts.append(frame)
        return pd.concat(parts, ignore_index=True)
    return load_hf_cache(dataset_id, RAW_DIR / f"{cache_name}.parquet", build)


def bipia():
    cols = ["id", "task_name", "context", "attack_name", "attack_str", "attack_category", "position"]
    return load_hf_cache(HF_BIPIA, RAW_DIR / "bipia.parquet",
                         lambda: load_dataset(HF_BIPIA, split="train").to_pandas()[cols])


def wikipedia_articles():
    def build():
        stream = load_dataset(HF_WIKIPEDIA, WIKIPEDIA_CONFIG, split="train", streaming=True)
        rows = []
        for row in stream:
            rows.append({"id": row["id"], "title": row["title"], "text": row["text"]})
            if len(rows) >= WIKIPEDIA_MAX_ARTICLES:
                break
        return pd.DataFrame(rows)
    return load_hf_cache(f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", RAW_DIR / f"wikipedia_{WIKIPEDIA_CONFIG}_first{WIKIPEDIA_MAX_ARTICLES}.parquet", build)


def email_ham():
    def build():
        dataset = load_dataset(HF_EMAIL, split="train")
        kept = dataset.filter(lambda x: x.get("label") == 0 and isinstance(x.get("text"), str) and len(x["text"].strip()) >= EMAIL_MIN_BODY_CHARS)
        return kept.to_pandas()[["text", "subject", "dataset_name"]]
    return load_hf_cache(f"{HF_EMAIL}:label0", RAW_DIR / "email_ham.parquet", build)


# ---------------- v4 loaders ----------------
from huggingface_hub import hf_hub_download

LLMAIL_WRAPPER_RE = re.compile(r"^Subject of the email:\s*(.*?)\.?\s{2,}Body:\s*", re.S)


def llmail_render(text):
    # "Subject of the email: X.   Body: Y" -> "Subject: X\n\nY", the layout email_ham uses, so the
    # challenge wrapper cannot become the signal. Rows without the wrapper are returned unchanged.
    m = LLMAIL_WRAPPER_RE.match(text)
    if not m:
        return text.strip()
    subject = m.group(1).strip()
    body = text[m.end():].strip()
    return f"Subject: {subject}\n\n{body}" if subject else body


def llmail_attacks():
    def build():
        frames = []
        for phase in ("phase1", "phase2"):
            path = hf_hub_download(HF_LLMAIL, f"data/labelled_unique_submissions_{phase}.json", repo_type="dataset")
            labelled = json.loads(Path(path).read_text(encoding="utf-8"))
            frame = pd.DataFrame({"raw": list(labelled.keys()),
                                  "attack_attempt": [str(v.get("attack_attempt")) for v in labelled.values()],
                                  "reason": [str(v.get("reason")) for v in labelled.values()]})
            frame["phase"] = phase
            frames.append(frame[(frame["attack_attempt"] == "True") & (frame["reason"] == "api_triggered")])
        frame = pd.concat(frames, ignore_index=True)
        frame = frame[frame["raw"].str.len() <= LLMAIL_MAX_CHARS]
        frame["text"] = frame["raw"].map(llmail_render)
        return frame[["text", "phase", "reason"]].reset_index(drop=True)
    return load_hf_cache(f"{HF_LLMAIL}:api_triggered", RAW_DIR / "llmail_api_triggered.parquet", build)


def llmail_clean_emails():
    def build():
        path = hf_hub_download(HF_LLMAIL, "data/emails_for_fp_tests.json", repo_type="dataset")
        emails = json.loads(Path(path).read_text(encoding="utf-8"))
        return pd.DataFrame({"text": [llmail_render(e) for e in emails]})
    return load_hf_cache(f"{HF_LLMAIL}:fp_emails", RAW_DIR / "llmail_fp_emails.parquet", build)


def piarena():
    def build():
        dd = load_dataset(HF_PIARENA)
        parts = []
        for task, part in dd.items():
            frame = part.to_pandas()[["context", "injected_task", "category"]]
            frame["task"] = task
            parts.append(frame)
        return pd.concat(parts, ignore_index=True)
    return load_hf_cache(HF_PIARENA, RAW_DIR / "piarena.parquet", build)


def nemotron_ipi():
    # datasets 4.x cannot parse this file's feature types; read the JSONL directly.
    def build():
        path = hf_hub_download(HF_NEMOTRON_IPI, "train.jsonl", repo_type="dataset")
        records = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
        out = []
        for rec in records:
            vector = rec.get("injection_vector")
            injection_text = (rec.get("injection") or {}).get("injection_text") or ""
            found = []

            def walk(node):
                if isinstance(node, dict):
                    for key, value in node.items():
                        if key == vector and isinstance(value, str):
                            found.append(value)
                        else:
                            walk(value)
                elif isinstance(node, list):
                    for value in node:
                        walk(value)
            walk(rec.get("environment"))
            for value in found:
                injected = bool(injection_text) and injection_text in value
                out.append({"record_id": str(rec.get("id")), "domain": rec.get("domain"), "attack_category": rec.get("attack_category"),
                            "vector": vector, "text": value, "injected": injected})
        frame = pd.DataFrame(out)
        assert frame.groupby("record_id")["injected"].any().all(), "a Nemotron IPI record has no field containing its injection text"
        return frame
    return load_hf_cache(HF_NEMOTRON_IPI, RAW_DIR / "nemotron_agentic_ipi.parquet", build)


def k8s():
    def build():
        frame = load_dataset(HF_K8S, split="train").to_pandas()[["text", "label", "vector", "manifest_id", "kind", "repo"]]
        frame = frame[frame["text"].str.len() <= K8S_MAX_CHARS]
        complete = frame.groupby("manifest_id")["label"].nunique()
        frame = frame[frame["manifest_id"].isin(complete[complete == 2].index)]
        keep = pd.Series(frame["manifest_id"].unique()).sample(n=min(K8S_MAX_PAIRS, frame["manifest_id"].nunique()), random_state=SEED)
        return frame[frame["manifest_id"].isin(set(keep))].reset_index(drop=True)
    return load_hf_cache(HF_K8S, RAW_DIR / "k8s_resource_injection.parquet", build)


def three_split_text_dataset(dataset_id, cache_name, columns):
    def build():
        dd = load_dataset(dataset_id)
        parts = []
        for split_name, part in dd.items():
            frame = part.to_pandas()[columns]
            frame["hf_split"] = split_name
            parts.append(frame)
        return pd.concat(parts, ignore_index=True)
    return load_hf_cache(dataset_id, RAW_DIR / f"{cache_name}.parquet", build)


def itw(config):
    return load_hf_cache(f"{HF_ITW}/{config}", RAW_DIR / f"itw_{config}.parquet",
                         lambda: load_dataset(HF_ITW, config, split="train").to_pandas()[["prompt", "platform", "source"]])


def notinject():
    def build():
        dd = load_dataset(HF_NOTINJECT)
        parts = []
        for subset, part in dd.items():
            frame = part.to_pandas()[["prompt", "category"]]
            frame["subset"] = subset
            parts.append(frame)
        return pd.concat(parts, ignore_index=True)
    return load_hf_cache(HF_NOTINJECT, RAW_DIR / "notinject.parquet", build)


raw = {
    "llmail_attacks": llmail_attacks(),
    "llmail_clean": llmail_clean_emails(),
    "piarena": piarena(),
    "nemotron_ipi": nemotron_ipi(),
    "k8s": k8s(),
    "agentic5k": three_split_text_dataset(HF_AGENTIC5K, "agentic5k", ["text", "label", "attack_family", "technique"]),
    "boundary": three_split_text_dataset(HF_BOUNDARY, "boundary_pairs", ["text", "label", "attack_family", "pair_id", "scenario_id", "source_context"]),
    "english3k": three_split_text_dataset(HF_ENGLISH3K, "english3k", ["text", "label", "attack_family", "technique"]),
    "itw_jailbreak": itw("jailbreak_2023_12_25"),
    "itw_regular": itw("regular_2023_12_25"),
    "ogma": load_hf_cache(HF_OGMA, RAW_DIR / "ogma_benign.parquet", lambda: load_dataset(HF_OGMA, split="train").to_pandas()[["text", "subtype", "length"]]),
    "slabs": three_split_text_dataset(HF_SLABS, "slabs", ["text", "label"]),
    "notinject": notinject(),
    "email_ham": email_ham(),
    "nemotron_train": nemotron_jailbreaking("train"),
    "nemotron_test": nemotron_jailbreaking("test"),
    "wildguard_train": wildguard("wildguardtrain", "train"),
    "wildguard_test": wildguard("wildguardtest", "test"),
    "deepset": with_splits(HF_DEEPSET, "deepset_prompt_injections"),
    "jackhhao": with_splits(HF_JACKHHAO, "jackhhao_jailbreak_classification"),
    "bipia": bipia(),
    "wikipedia": wikipedia_articles(),
}
for name, frame in raw.items():
    print(f"{name:16s} {len(frame):>7,} rows  columns={frame.columns.tolist()}")


microsoft/llmail-inject-challenge:api_triggered (cached): 21,177 rows
microsoft/llmail-inject-challenge:fp_emails (cached): 203 rows
sleeepeer/PIArena (cached): 2,000 rows
nvidia/Nemotron-RL-Agentic-Indirect-Prompt-Injection-v1 (cached): 600 rows
rgeada/k8s-resource-prompt-injection (cached): 5,000 rows
3nesdeniz/agentic-prompt-injection-5k (cached): 4,935 rows
3nesdeniz/agentic-prompt-injection-boundary-pairs (cached): 1,200 rows
3nesdeniz/english-prompt-injection-3k (cached): 3,053 rows
TrustAIRLab/in-the-wild-jailbreak-prompts/jailbreak_2023_12_25 (cached): 1,405 rows
TrustAIRLab/in-the-wild-jailbreak-prompts/regular_2023_12_25 (cached): 13,735 rows
axiotic/ogma-prompt-injection-benign (cached): 11,368 rows
S-Labs/prompt-injection-dataset (cached): 15,291 rows
leolee99/NotInject (cached): 339 rows


puyang2025/seven-phishing-email-datasets:label0 (cached): 80,036 rows
nvidia/Nemotron-Safety-Guard-Dataset-v3:train:jailbreaking:en (cached): 10,000 rows
nvidia/Nemotron-Safety-Guard-Dataset-v3:test:jailbreaking:en (cached): 1,000 rows
allenai/wildguardmix/wildguardtrain (cached): 86,759 rows
allenai/wildguardmix/wildguardtest (cached): 1,725 rows
deepset/prompt-injections (cached): 662 rows
jackhhao/jailbreak-classification (cached): 1,306 rows


geodesic-research/bipia (cached): 41,250 rows
wikimedia/wikipedia/20231101.en (cached): 1,500 rows
llmail_attacks    21,177 rows  columns=['text', 'phase', 'reason']
llmail_clean         203 rows  columns=['text']
piarena            2,000 rows  columns=['context', 'injected_task', 'category', 'task']
nemotron_ipi         600 rows  columns=['record_id', 'domain', 'attack_category', 'vector', 'text', 'injected']
k8s                5,000 rows  columns=['text', 'label', 'vector', 'manifest_id', 'kind', 'repo']
agentic5k          4,935 rows  columns=['text', 'label', 'attack_family', 'technique', 'hf_split']
boundary           1,200 rows  columns=['text', 'label', 'attack_family', 'pair_id', 'scenario_id', 'source_context', 'hf_split']
english3k          3,053 rows  columns=['text', 'label', 'attack_family', 'technique', 'hf_split']
itw_jailbreak      1,405 rows  columns=['prompt', 'platform', 'source']
itw_regular       13,735 rows  columns=['prompt', 'platform', 'source']
ogma            

## 4. Normalize

Every source is mapped to `text, label, reason, source, shape, synthetic, attack_type, hf_dataset,
source_id, group_key`. `shape` is source-derived (chat prompt vs document chunk), recorded so the
split allocator can balance it and the evaluation can slice on it. `group_key` is set only where a
source has an explicit template or document identity (BIPIA, keyed by normalized text); everything else is grouped by
near-duplicate cluster and leading-words template key in section 7.


In [4]:
COMMON_COLS = ["text", "label", "reason", "source", "shape", "synthetic", "attack_type", "hf_dataset", "source_id", "group_key", "forced_split", "doc_key"]


def values(x):
    # Series are taken by position, never aligned by index: the source frames carry arbitrary indices.
    return x.to_numpy() if isinstance(x, pd.Series) else x


def rows(frame, text_col, label, reason, source, shape, hf_dataset, source_id, attack_type=None, group_key=None, forced_split=None, doc_key=None):
    out = pd.DataFrame({
        "text": frame[text_col].fillna("").astype(str).to_numpy(),
        "label": int(label),
        "reason": reason if label else "",
        "source": source,
        "shape": shape,
        "synthetic": False,
        "attack_type": values(attack_type) if attack_type is not None else "",
        "hf_dataset": hf_dataset,
        "source_id": values(source_id),
        "group_key": values(group_key) if group_key is not None else None,
        "forced_split": values(forced_split) if forced_split is not None else None,
        "doc_key": values(doc_key) if doc_key is not None else None,
    })
    return out[COMMON_COLS]


def seeded_cap(frame, cap):
    return frame if not cap or len(frame) <= cap else frame.sample(n=cap, random_state=SEED)


parts = []

# Nemotron: every jailbreaking row is INJECTION (LoRA labelling); prompt_label kept for the slice.
nm = seeded_cap(raw["nemotron_train"], NEMOTRON_MAX)
parts.append(rows(nm, "prompt", 1, "Jailbreak Technique", "nemotron_jailbreaking", "chat", HF_NEMOTRON,
                  "nemotron:" + nm["id"].astype(str), attack_type="jailbreaking/" + nm["prompt_label"].fillna("").astype(str)))

# WildGuardMix train. Adversarial + harmful -> INJECTION with the LoRA's subcategory map.
# Unharmful -> SAFE, split into two source slices by the adversarial flag.
wg = raw["wildguard_train"].copy()
wg["source_id"] = "wildguardtrain:" + wg.index.astype(str)
wg_inj = seeded_cap(wg[(wg["adversarial"] == True) & (wg["prompt_harm_label"] == "harmful")], WILDGUARD_INJECTION_MAX)
wg_inj_reason = wg_inj["subcategory"].fillna("others").map(lambda s: SUBCATEGORY_REASON_MAP.get(s, DEFAULT_REASON))
frame = rows(wg_inj, "prompt", 1, "", "wildguard_adversarial", "chat", f"{HF_WILDGUARD}/wildguardtrain",
             wg_inj["source_id"], attack_type=wg_inj["subcategory"].fillna("others"))
frame["reason"] = wg_inj_reason.to_numpy()
parts.append(frame)
wg_safe = seeded_cap(wg[wg["prompt_harm_label"] == "unharmful"], WILDGUARD_SAFE_MAX)
for adversarial, source in ((False, "wildguard_benign"), (True, "wildguard_benign_adversarial")):
    sub = wg_safe[wg_safe["adversarial"] == adversarial]
    parts.append(rows(sub, "prompt", 0, "", source, "chat", f"{HF_WILDGUARD}/wildguardtrain", sub["source_id"],
                      attack_type=sub["subcategory"].fillna("benign")))

# deepset: short direct injections (en/de). Positives are overrides of the preceding instructions.
dp = raw["deepset"].copy()
dp["source_id"] = "deepset:" + dp["hf_split"] + ":" + dp.index.astype(str)
parts.append(rows(dp[dp["label"] == 1], "text", 1, "Override Attempt", "deepset_injection", "chat", HF_DEEPSET, dp.loc[dp["label"] == 1, "source_id"]))
parts.append(rows(dp[dp["label"] == 0], "text", 0, "", "deepset_benign", "chat", HF_DEEPSET, dp.loc[dp["label"] == 0, "source_id"]))

# jackhhao: DAN-style jailbreak prompts vs benign role-play prompts.
jh = raw["jackhhao"].copy()
jh["source_id"] = "jackhhao:" + jh["hf_split"] + ":" + jh.index.astype(str)
parts.append(rows(jh[jh["type"] == "jailbreak"], "prompt", 1, "Jailbreak Technique", "jackhhao_jailbreak", "chat", HF_JACKHHAO, jh.loc[jh["type"] == "jailbreak", "source_id"]))
parts.append(rows(jh[jh["type"] == "benign"], "prompt", 0, "", "jackhhao_benign", "chat", HF_JACKHHAO, jh.loc[jh["type"] == "benign", "source_id"]))

# BIPIA: the indirect case. `context` already carries `attack_str` at `position` (asserted here).
# Injected rows are grouped by attack template so a template graded in test was never trained on.
# Clean documents are recovered by removing the attack string once; one SAFE row per base document.
bp = raw["bipia"].copy()
bp["context"] = bp["context"].fillna("").astype(str)
bp["attack_str"] = bp["attack_str"].fillna("").astype(str)
assert (bp["attack_str"].str.len() > 0).all(), "BIPIA row without an attack string"
inside = [a in c for a, c in zip(bp["attack_str"], bp["context"])]
assert all(inside), f"{len(inside) - sum(inside)} BIPIA rows do not contain their attack string in context"
bp["base_doc"] = [c.replace(a, "", 1).strip() for a, c in zip(bp["attack_str"], bp["context"])]
bp["template_id"] = bp["attack_str"].map(short_hash)
bp["doc_id"] = bp["base_doc"].map(lambda t: short_hash(normalize_for_fp(t)))  # v3: normalized, so start/middle/end variants share one id


def partition(keys, seed):
    # Independent, seeded 80/10/10 assignment of a set of ids to splits.
    keys = sorted(set(keys))
    order = np.random.default_rng(seed).permutation(len(keys))
    bounds = np.cumsum([SPLITS[s] for s in SPLITS]) * len(keys)
    assignment = {}
    for rank, index in enumerate(order):
        split = next(s for s, bound in zip(SPLITS, bounds) if rank < bound - 1e-9) if rank < bounds[-1] else list(SPLITS)[-1]
        assignment[keys[index]] = split
    return assignment


doc_split = partition(bp["doc_id"], BIPIA_SPLIT_SEED)
template_split = partition(bp["template_id"], BIPIA_SPLIT_SEED + 1)
bp["doc_split"] = bp["doc_id"].map(doc_split)
bp["template_split"] = bp["template_id"].map(template_split)
# v2: an injected row is usable only when its document and its attack string landed in the same
# split. Test rows therefore carry an unseen document and an unseen attack string.
bp_same = bp[bp["doc_split"] == bp["template_split"]]
bp_inj = bp_same.sample(frac=1.0, random_state=SEED).groupby("doc_id").head(BIPIA_ROWS_PER_DOC)
parts.append(rows(bp_inj, "context", 1, "Embedded Instruction", "bipia_injected", "document", HF_BIPIA, "bipia:" + bp_inj["id"].astype(str),
                  attack_type=bp_inj["task_name"] + "/" + bp_inj["attack_category"] + "/" + bp_inj["attack_name"] + "/" + bp_inj["position"],
                  group_key="bipia_template:" + bp_inj["template_id"], forced_split=bp_inj["template_split"], doc_key="bipia_doc:" + bp_inj["doc_id"]))
bp_clean = bp.drop_duplicates("doc_id")
parts.append(rows(bp_clean, "base_doc", 0, "", "bipia_clean", "document", HF_BIPIA, "bipia_clean:" + bp_clean["doc_id"],
                  attack_type=bp_clean["task_name"], group_key="bipia_doc:" + bp_clean["doc_id"], forced_split=bp_clean["doc_split"], doc_key="bipia_doc:" + bp_clean["doc_id"]))
print(f"BIPIA: {bp['template_id'].nunique()} attack templates, {bp['doc_id'].nunique()} base documents; "
      f"{len(bp_same):,} rows with document and template in the same split, {len(bp_inj):,} injected rows sampled, {len(bp_clean):,} clean documents")
print("BIPIA injected rows per split:", bp_inj["template_split"].value_counts().to_dict(), "| clean docs per split:", bp_clean["doc_split"].value_counts().to_dict())

# Legitimate emails: structured, document-shaped SAFE rows that are not BIPIA, so document shape
# alone, and BIPIA's house style in particular, cannot stand in for "contains an attack".
em = raw["email_ham"].copy()
em["text"] = em["text"].fillna("").astype(str)
em["subject"] = em["subject"].fillna("").astype(str)
em["doc"] = np.where(em["subject"].str.strip().str.len() > 0, "Subject: " + em["subject"].str.strip() + "\n\n" + em["text"], em["text"])
em = seeded_cap(em, EMAIL_HAM_MAX)
em["source_id"] = "email_ham:" + em.index.astype(str)
parts.append(rows(em, "doc", 0, "", "email_ham", "document", HF_EMAIL, em["source_id"], attack_type=em["dataset_name"].fillna("").astype(str)))

# Wikipedia: prose-shaped SAFE paragraphs so document shape alone is not an injection signal.
wk_rows = []
for _, article in raw["wikipedia"].iterrows():
    for k, paragraph in enumerate(str(article["text"]).split("\n\n")):
        paragraph = paragraph.strip()
        if len(paragraph) >= WIKIPEDIA_MIN_PARAGRAPH_CHARS and not paragraph.startswith("=="):
            wk_rows.append({"text": paragraph, "source_id": f"wikipedia:{article['id']}#{k}", "doc": str(article["id"])})
wk = pd.DataFrame(wk_rows)
wk = wk.iloc[np.random.default_rng(SEED).permutation(len(wk))[:WIKIPEDIA_MAX_PARAGRAPHS]]
parts.append(rows(wk, "text", 0, "", "wikipedia_paragraph", "document", f"{HF_WIKIPEDIA}/{WIKIPEDIA_CONFIG}", wk["source_id"],
                  group_key="wikipedia_article:" + wk["doc"]))

# ---------------- v4 sources ----------------
# LLMail: real adaptive attack emails whose injected tool call fired (api_triggered), re-rendered to the
# email_ham layout. Reason: an instruction embedded in a document. The 203 clean emails the challenge used
# for its own false-positive tests are the matched SAFE rows.
lm = seeded_cap(raw["llmail_attacks"], LLMAIL_MAX).reset_index(drop=True)
lm["source_id"] = "llmail:" + lm["phase"] + ":" + lm.index.astype(str)
parts.append(rows(lm, "text", 1, "Embedded Instruction", "llmail_injected", "document", HF_LLMAIL, lm["source_id"], attack_type="llmail/" + lm["phase"] + "/" + lm["reason"]))
lc = raw["llmail_clean"].reset_index(drop=True)
parts.append(rows(lc, "text", 0, "", "llmail_clean", "document", HF_LLMAIL, "llmail_clean:" + lc.index.astype(str)))

# PIArena: each clean passage is a SAFE document; the same passage with the injected task planted at a
# seeded sentence boundary is the INJECTION twin. Both share a document group key so they never straddle splits.
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def cut_at_sentence(text, max_chars):
    text = WS_RE.sub(" ", text).strip()
    if len(text) <= max_chars:
        return text
    sentences = SENTENCE_SPLIT_RE.split(text)
    out, total = [], 0
    for s in sentences:
        if total + len(s) + 1 > max_chars and out:
            break
        out.append(s)
        total += len(s) + 1
    return " ".join(out) if out else text[:max_chars]


def plant(text, payload, seed):
    sentences = SENTENCE_SPLIT_RE.split(text)
    k = int(np.random.default_rng(seed).integers(0, len(sentences) + 1))
    return " ".join(sentences[:k] + [payload.strip()] + sentences[k:]).strip()


pa = raw["piarena"].copy()
pa["context"] = pa["context"].fillna("").astype(str).map(lambda t: cut_at_sentence(t, PIARENA_MAX_CONTEXT_CHARS))
pa["injected_task"] = pa["injected_task"].fillna("").astype(str)
pa = pa[(pa["context"].str.len() >= 50) & (pa["injected_task"].str.len() > 0)].drop_duplicates("context").reset_index(drop=True)
pa["doc_id"] = pa["context"].map(lambda t: short_hash(normalize_for_fp(t)))
pa["planted"] = [plant(c, t, SEED + i) for i, (c, t) in enumerate(zip(pa["context"], pa["injected_task"]))]
parts.append(rows(pa, "planted", 1, "Embedded Instruction", "piarena_injected", "document", HF_PIARENA, "piarena:" + pa["task"] + ":" + pa.index.astype(str),
                  attack_type="piarena/" + pa["task"] + "/" + pa["category"].astype(str), group_key="piarena_doc:" + pa["doc_id"], doc_key="piarena_doc:" + pa["doc_id"]))
parts.append(rows(pa, "context", 0, "", "piarena_clean", "document", HF_PIARENA, "piarena_clean:" + pa["task"] + ":" + pa.index.astype(str),
                  attack_type="piarena/" + pa["task"], group_key="piarena_doc:" + pa["doc_id"], doc_key="piarena_doc:" + pa["doc_id"]))

# Nemotron agentic IPI: the record field that carries the injection is the INJECTION document; every
# other value of that field name in the same record (other patients' chart notes, other products'
# descriptions) is a clean document of the same kind. One group per record.
ni = raw["nemotron_ipi"].copy()
ni = ni[ni["text"].str.strip().str.len() >= MIN_TEXT_CHARS].reset_index(drop=True)
ni_inj = ni[ni["injected"]]
ni_clean = ni[~ni["injected"]]
parts.append(rows(ni_inj, "text", 1, "Embedded Instruction", "nemotron_ipi_injected", "document", HF_NEMOTRON_IPI, "nipi:" + ni_inj["record_id"] + ":" + ni_inj.index.astype(str),
                  attack_type="nipi/" + ni_inj["domain"].astype(str) + "/" + ni_inj["attack_category"].astype(str) + "/" + ni_inj["vector"].astype(str),
                  group_key="nipi_record:" + ni_inj["record_id"], doc_key="nipi_record:" + ni_inj["record_id"]))
parts.append(rows(ni_clean, "text", 0, "", "nemotron_ipi_clean", "document", HF_NEMOTRON_IPI, "nipi_clean:" + ni_clean["record_id"] + ":" + ni_clean.index.astype(str),
                  attack_type="nipi/" + ni_clean["domain"].astype(str) + "/" + ni_clean["vector"].astype(str),
                  group_key="nipi_record:" + ni_clean["record_id"], doc_key="nipi_record:" + ni_clean["record_id"]))

# Kubernetes manifests: real resource files, each in a clean and an injected version. One group per manifest.
k8 = raw["k8s"].copy().reset_index(drop=True)
k8["source_id"] = "k8s:" + k8["manifest_id"].astype(str) + ":" + k8["label"].astype(str)
k8_inj = k8[k8["label"] == "injection"]
k8_clean = k8[k8["label"] == "safe"]
parts.append(rows(k8_inj, "text", 1, "Embedded Instruction", "k8s_injected", "document", HF_K8S, k8_inj["source_id"],
                  attack_type="k8s/" + k8_inj["vector"].astype(str) + "/" + k8_inj["kind"].astype(str), group_key="k8s_manifest:" + k8_inj["manifest_id"].astype(str), doc_key="k8s_manifest:" + k8_inj["manifest_id"].astype(str)))
parts.append(rows(k8_clean, "text", 0, "", "k8s_clean", "document", HF_K8S, k8_clean["source_id"],
                  attack_type="k8s/" + k8_clean["kind"].astype(str), group_key="k8s_manifest:" + k8_clean["manifest_id"].astype(str), doc_key="k8s_manifest:" + k8_clean["manifest_id"].astype(str)))

# 3nesdeniz agentic 5k: attacks and matched hard negatives on the agentic surface. Family -> reason;
# the `delimiter_confusion` technique overrides its family. The dataset's own splits are ignored; our
# grouping (near-dup + leading-words template) decides.
a5 = raw["agentic5k"].copy().reset_index(drop=True)
a5["label"] = pd.to_numeric(a5["label"], errors="coerce").astype(int)
a5["source_id"] = "agentic5k:" + a5["hf_split"] + ":" + a5.index.astype(str)
a5_inj = a5[a5["label"] == 1]
a5_reason = [AGENTIC5K_TECHNIQUE_REASON.get(str(t), AGENTIC5K_FAMILY_REASON.get(str(f), DEFAULT_REASON)) for f, t in zip(a5_inj["attack_family"], a5_inj["technique"])]
frame = rows(a5_inj, "text", 1, "", "agentic5k_injection", "agentic", HF_AGENTIC5K, a5_inj["source_id"], attack_type=a5_inj["attack_family"].astype(str) + "/" + a5_inj["technique"].astype(str))
frame["reason"] = a5_reason
parts.append(frame)
a5_ben = a5[a5["label"] == 0]
parts.append(rows(a5_ben, "text", 0, "", "agentic5k_benign", "agentic", HF_AGENTIC5K, a5_ben["source_id"], attack_type=a5_ben["technique"].astype(str)))

# 3nesdeniz boundary pairs: each attack has a benign twin from the same workflow. Shape from the
# source context; one group per scenario so a workflow is graded only where it was never trained.
bd = raw["boundary"].copy().reset_index(drop=True)
bd["label"] = pd.to_numeric(bd["label"], errors="coerce").astype(int)
bd["shape"] = bd["source_context"].astype(str).map(lambda c: BOUNDARY_CONTEXT_SHAPE.get(c, "document"))
bd["source_id"] = "boundary:" + bd["pair_id"].astype(str) + ":" + bd.index.astype(str)
# Scenario-isolated like the dataset itself: a seeded 80/10/10 partition of the 50 scenario ids is
# forced, so test is guaranteed its share. Without this the greedy allocator placed every 240-row
# scenario group in train or val (first v4 run: 0 boundary rows in test). Boundary rows are excluded
# from LSH and template merging (sections 6 and 7) so a group is exactly one scenario.
bd["forced_split"] = bd["scenario_id"].astype(str).map(partition(bd["scenario_id"].astype(str), BIPIA_SPLIT_SEED + 2))
for shape_name in sorted(bd["shape"].unique()):
    sub = bd[bd["shape"] == shape_name]
    sub_inj = sub[sub["label"] == 1]
    frame = rows(sub_inj, "text", 1, "", "boundary_injection", shape_name, HF_BOUNDARY, sub_inj["source_id"],
                 attack_type=sub_inj["attack_family"].astype(str) + "/" + sub_inj["source_context"].astype(str), group_key="boundary_scenario:" + sub_inj["scenario_id"].astype(str),
                 forced_split=sub_inj["forced_split"])
    frame["reason"] = sub_inj["attack_family"].astype(str).map(lambda f: BOUNDARY_FAMILY_REASON.get(f, DEFAULT_REASON)).to_numpy()
    parts.append(frame)
    sub_ben = sub[sub["label"] == 0]
    parts.append(rows(sub_ben, "text", 0, "", "boundary_benign", shape_name, HF_BOUNDARY, sub_ben["source_id"],
                      attack_type=sub_ben["source_context"].astype(str), group_key="boundary_scenario:" + sub_ben["scenario_id"].astype(str),
                      forced_split=sub_ben["forced_split"]))

# 3nesdeniz English 3k: direct attacks in eight families, with look-alike benign rows.
e3 = raw["english3k"].copy().reset_index(drop=True)
e3["label"] = pd.to_numeric(e3["label"], errors="coerce").astype(int)
e3["source_id"] = "english3k:" + e3["hf_split"] + ":" + e3.index.astype(str)
e3_inj = e3[e3["label"] == 1]
frame = rows(e3_inj, "text", 1, "", "english3k_injection", "chat", HF_ENGLISH3K, e3_inj["source_id"], attack_type=e3_inj["attack_family"].astype(str) + "/" + e3_inj["technique"].astype(str))
frame["reason"] = e3_inj["attack_family"].astype(str).map(lambda f: ENGLISH3K_FAMILY_REASON.get(f, DEFAULT_REASON)).to_numpy()
parts.append(frame)
e3_ben = e3[e3["label"] == 0]
parts.append(rows(e3_ben, "text", 0, "", "english3k_benign", "chat", HF_ENGLISH3K, e3_ben["source_id"], attack_type=e3_ben["technique"].astype(str)))

# In-the-wild prompts: real jailbreaks from Reddit, Discord and prompt sites, and real non-jailbreak
# prompt templates from the same places (instruction-heavy benign text).
ij = raw["itw_jailbreak"].copy()
ij = ij[ij["prompt"].fillna("").str.len() <= ITW_MAX_CHARS].reset_index(drop=True)
parts.append(rows(ij, "prompt", 1, "Jailbreak Technique", "itw_jailbreak", "chat", f"{HF_ITW}/jailbreak_2023_12_25", "itw_jb:" + ij.index.astype(str), attack_type="itw/" + ij["platform"].astype(str) + "/" + ij["source"].astype(str)))
ir = raw["itw_regular"].copy()
ir = seeded_cap(ir[ir["prompt"].fillna("").str.len() <= ITW_MAX_CHARS], ITW_REGULAR_MAX).reset_index(drop=True)
parts.append(rows(ir, "prompt", 0, "", "itw_regular_benign", "chat", f"{HF_ITW}/regular_2023_12_25", "itw_reg:" + ir.index.astype(str), attack_type="itw/" + ir["platform"].astype(str) + "/" + ir["source"].astype(str)))

# Everyday benign requests, shell commands and code (ogma); AI-safety questions and other hard negatives (S-Labs benign).
og = seeded_cap(raw["ogma"], OGMA_MAX).reset_index(drop=True)
parts.append(rows(og, "text", 0, "", "ogma_benign", "chat", HF_OGMA, "ogma:" + og.index.astype(str), attack_type="ogma/" + og["subtype"].astype(str) + "/" + og["length"].astype(str)))
sl = raw["slabs"].copy()
sl = seeded_cap(sl[pd.to_numeric(sl["label"], errors="coerce") == 0], SLABS_BENIGN_MAX).reset_index(drop=True)
parts.append(rows(sl, "text", 0, "", "slabs_benign", "chat", HF_SLABS, "slabs:" + sl["hf_split"] + ":" + sl.index.astype(str)))

# NotInject: benign prompts built from trigger words. Forced into test so over-defense is measured, never learned.
nj = raw["notinject"].copy().reset_index(drop=True)
parts.append(rows(nj, "prompt", 0, "", "notinject_benign", "chat", HF_NOTINJECT, "notinject:" + nj["subset"].astype(str) + ":" + nj.index.astype(str),
                  attack_type="notinject/" + nj["subset"].astype(str) + "/" + nj["category"].astype(str), forced_split=pd.Series(["test"] * len(nj))))

df = pd.concat(parts, ignore_index=True)
df["text"] = df["text"].fillna("").astype(str)
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df["holdout"] = False

# Held-out comparison set, labelled exactly as safety/notebooks/prompt_injection_lora_qwen38_27b_eval.ipynb labels it.
wt = raw["wildguard_test"].copy()
wt["source_id"] = "wildguardtest:" + wt.index.astype(str)
wt_safe = wt[wt["prompt_harm_label"] == "unharmful"]
wt_inj = wt[(wt["adversarial"] == True) & (wt["prompt_harm_label"] == "harmful")]
holdout_parts = [
    rows(wt_safe, "prompt", 0, "", "wildguard_test", "chat", f"{HF_WILDGUARD}/wildguardtest", wt_safe["source_id"], attack_type=wt_safe["subcategory"].fillna("benign")),
    rows(wt_inj, "prompt", 1, "", "wildguard_test", "chat", f"{HF_WILDGUARD}/wildguardtest", wt_inj["source_id"], attack_type=wt_inj["subcategory"].fillna("others")),
]
holdout_parts[1]["reason"] = wt_inj["subcategory"].fillna("others").map(lambda s: SUBCATEGORY_REASON_MAP.get(s, DEFAULT_REASON)).to_numpy()
nt = raw["nemotron_test"]
holdout_parts.append(rows(nt, "prompt", 1, "Jailbreak Technique", "nemotron_test_jailbreaking", "chat", HF_NEMOTRON,
                          "nemotron_test:" + nt["id"].astype(str), attack_type="jailbreaking/" + nt["prompt_label"].fillna("").astype(str)))
holdout = pd.concat(holdout_parts, ignore_index=True)
holdout["text"] = holdout["text"].fillna("").astype(str)
holdout["holdout"] = True

print(df.groupby(["source", "label"]).size().unstack(fill_value=0))
print()
print("shape x label:")
print(df.groupby(["shape", "label"]).size().unstack(fill_value=0))
print()
print("held-out rows:", holdout.groupby(["source", "label"]).size().to_dict())
print(f"pool rows before quality gates: {len(df):,}")


BIPIA: 125 attack templates, 182 base documents; 27,514 rows with document and template in the same split, 546 injected rows sampled, 182 clean documents
BIPIA injected rows per split: {'train': 438, 'val': 54, 'test': 54} | clean docs per split: {'train': 146, 'test': 18, 'val': 18}


label                             0      1
source                                    
agentic5k_benign               2185      0
agentic5k_injection               0   2750
bipia_clean                     182      0
bipia_injected                    0    546
boundary_benign                 600      0
boundary_injection                0    600
deepset_benign                  399      0
deepset_injection                 0    263
email_ham                      3000      0
english3k_benign               1128      0
english3k_injection               0   1925
itw_jailbreak                     0   1309
itw_regular_benign             5000      0
jackhhao_benign                 640      0
jackhhao_jailbreak                0    666
k8s_clean                      2500      0
k8s_injected                      0   2500
llmail_clean                    203      0
llmail_injected                   0   8000
nemotron_ipi_clean              371      0
nemotron_ipi_injected             0    229
nemotron_ja

## 5. Quality Gates

Drop rows with an invalid label, a reason outside the contract, or near-empty text. Applied to the pool and to the held-out set.


In [5]:
def quality_gate(frame, name):
    n0 = len(frame)
    frame = frame[frame["label"].isin([0, 1])].copy()
    frame["label"] = frame["label"].astype(int)
    valid_reason = ((frame["label"] == 0) & (frame["reason"] == "")) | ((frame["label"] == 1) & frame["reason"].isin(REASONS))
    bad = int((~valid_reason).sum())
    assert bad == 0, f"{name}: {bad} rows violate the reason contract"
    frame = frame[frame["text"].str.strip().str.len() >= MIN_TEXT_CHARS].copy()
    frame = frame.reset_index(drop=True)
    print(f"{name} quality gates: {n0:,} -> {len(frame):,} rows ({n0 - len(frame):,} dropped)")
    return frame


df = quality_gate(df, "pool")
holdout = quality_gate(holdout, "holdout")


pool quality gates: 124,775 -> 124,463 rows (312 dropped)
holdout quality gates: 2,286 -> 2,286 rows (0 dropped)


## 6. Deduplication and Held-Out Isolation

1. **Exact**: SHA-256 of whitespace-normalized, lowercased text. Pool rows that match a held-out
   row are dropped.
2. **Near-dup**: MinHash LSH over word 5-gram shingles, clustered with union-find, built over the
   pool **and** the held-out rows together. Any pool row that lands in a cluster with a held-out row
   is dropped, so the LoRA comparison set shares no paraphrase with training data. Clusters are
   later used as split-grouping keys.

BIPIA rows are excluded from the LSH: by construction every injected row is a near-duplicate of
its base document and of every other attack on that document, which would merge the corpus into one
cluster. They carry explicit template and document keys instead (section 4).


In [6]:
df["fp"] = df["text"].map(fingerprint)
holdout["fp"] = holdout["text"].map(fingerprint)
n0 = len(df)
df = df.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup (pool): {n0:,} -> {len(df):,}")
n0 = len(holdout)
holdout = holdout.drop_duplicates(subset="fp", keep="first").reset_index(drop=True)
print(f"exact dedup (holdout): {n0:,} -> {len(holdout):,}")
n0 = len(df)
df = df[~df["fp"].isin(set(holdout["fp"]))].reset_index(drop=True)
print(f"pool rows identical to a held-out row dropped: {n0 - len(df):,}")


def word_shingles(text, k=SHINGLE_SIZE):
    words = normalize_for_fp(text).split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}


combined = pd.concat([df[["text", "source", "holdout"]], holdout[["text", "source", "holdout"]]], ignore_index=True)
n_pool = len(df)
parent = list(range(len(combined)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=MINHASH_PERMS)
signatures = {}
# Paired document sources (clean twin + injected twin) are near-duplicates of each other by construction
# and carry explicit group keys instead; LSH over them would merge a whole source into one cluster.
skip_lsh = combined["source"].str.startswith(("bipia_", "piarena_", "nemotron_ipi_", "k8s_", "boundary_")).to_numpy()
for i, text in enumerate(combined["text"]):
    if skip_lsh[i]:
        continue
    mh = MinHash(num_perm=MINHASH_PERMS)
    for sh in word_shingles(text):
        mh.update(sh.encode("utf-8", "replace"))
    signatures[i] = mh
    lsh.insert(str(i), mh)
    if (i + 1) % 20000 == 0:
        print(f"  minhash {i + 1:,}/{len(combined):,}")

for i, mh in signatures.items():
    for key in lsh.query(mh):
        j = int(key)
        if j != i:
            union(i, j)

combined["dup_cluster"] = [find(i) for i in range(len(combined))]
holdout_clusters = set(combined.loc[combined["holdout"], "dup_cluster"])
pool_clusters = combined.loc[:n_pool - 1, "dup_cluster"].to_numpy()
near_holdout = np.isin(pool_clusters, list(holdout_clusters))
df["dup_cluster"] = pool_clusters
n0 = len(df)
df = df[~near_holdout].reset_index(drop=True)
print(f"pool rows that are near-duplicates of a held-out row dropped: {n0 - len(df):,}")
print(f"near-dup clusters (pool): {df['dup_cluster'].nunique():,} for {len(df):,} rows")
del signatures, lsh, combined, parent
gc.collect()


exact dedup (pool): 124,463 -> 93,761
exact dedup (holdout): 2,286 -> 2,086
pool rows identical to a held-out row dropped: 1


  minhash 20,000/95,846


  minhash 40,000/95,846


  minhash 60,000/95,846


  minhash 80,000/95,846


pool rows that are near-duplicates of a held-out row dropped: 0
near-dup clusters (pool): 90,891 for 93,760 rows


844967

## 7. Leakage-Safe Splits

Grouping key = union of (a) near-dup cluster, (b) explicit `group_key` where a source has one
(BIPIA template or document, Wikipedia article), and (c) for chat-shaped positives, the leading
`TEMPLATE_KEY_WORDS` words, so prompts generated from one jailbreak skeleton never straddle splits.
The ten largest groups are printed; a single giant group would make the split degenerate and is
something to look at, not hide.

Groups are assigned greedily (largest first, seeded shuffle among ties) by a marginal-cost rule that
balances rows, labels, every source, every reason, and both shapes against the 80/10/10 targets.
No group spans splits by construction.


In [7]:
parent = list(range(len(df)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


first_seen = {}
for i, key in enumerate(df["dup_cluster"]):
    key = ("cluster", int(key))
    if key in first_seen:
        union(i, first_seen[key])
    else:
        first_seen[key] = i
for i, key in enumerate(df["group_key"]):
    if isinstance(key, str) and key:
        key = ("explicit", key)
        if key in first_seen:
            union(i, first_seen[key])
        else:
            first_seen[key] = i
df["template_key"] = [template_key(t) if (lab == 1 and shape in ("chat", "agentic") and not str(source).startswith("boundary_")) else None
                      for t, lab, shape, source in zip(df["text"], df["label"], df["shape"], df["source"])]
for i, key in enumerate(df["template_key"]):
    if key:
        key = ("template", key)
        if key in first_seen:
            union(i, first_seen[key])
        else:
            first_seen[key] = i
df["group"] = [find(i) for i in range(len(df))]

largest = df.groupby("group").agg(n=("text", "size"), source=("source", lambda s: s.value_counts().index[0]),
                                  head=("text", lambda s: s.iloc[0][:70].replace("\n", " "))).sort_values("n", ascending=False).head(10)
print("largest groups:")
print(largest.to_string())
print()

SOURCES = sorted(df["source"].unique().tolist())
SHAPES = sorted(df["shape"].unique().tolist())
BALANCE_KEYS = (["rows", "pos", "neg"] + [f"src::{s}" for s in SOURCES]
                + [f"reason::{r}" for r in REASONS] + [f"shape::{s}" for s in SHAPES])

group_stats = {}
for gid, sub in df.groupby("group"):
    stats = {"rows": len(sub), "pos": int((sub["label"] == 1).sum()), "neg": int((sub["label"] == 0).sum())}
    src_counts = sub["source"].value_counts()
    reason_counts = sub["reason"].value_counts()
    shape_counts = sub["shape"].value_counts()
    for s in SOURCES:
        stats[f"src::{s}"] = int(src_counts.get(s, 0))
    for r in REASONS:
        stats[f"reason::{r}"] = int(reason_counts.get(r, 0))
    for s in SHAPES:
        stats[f"shape::{s}"] = int(shape_counts.get(s, 0))
    group_stats[gid] = stats

totals = {key: sum(stats[key] for stats in group_stats.values()) for key in BALANCE_KEYS}
targets = {split: {key: SPLITS[split] * totals[key] for key in BALANCE_KEYS} for split in SPLITS}
alloc = {split: {key: 0 for key in BALANCE_KEYS} for split in SPLITS}

order = list(group_stats)
np.random.default_rng(SEED).shuffle(order)
order.sort(key=lambda gid: -group_stats[gid]["rows"])


def marginal_cost(split, stats):
    cost = 0.0
    for key in BALANCE_KEYS:
        target = targets[split][key]
        if target <= 0:
            continue
        before = alloc[split][key] - target
        after = before + stats[key]
        cost += (after * after - before * before) / target
    return cost


assign = {}
# Groups with a pre-assigned split (BIPIA documents and templates, section 4) are placed first and
# counted toward the balance; a group must not contain two different forced splits.
forced = df.groupby("group")["forced_split"].agg(lambda s: set(x for x in s if isinstance(x, str)))
for gid, splits in forced.items():
    if splits:
        assert len(splits) == 1, f"group {gid} mixes forced splits {splits}"
        split = next(iter(splits))
        assign[gid] = split
        for key in BALANCE_KEYS:
            alloc[split][key] += group_stats[gid][key]
print(f"groups with a forced split: {len(assign):,}")
for gid in order:
    if gid in assign:
        continue
    stats = group_stats[gid]
    split = min(SPLITS, key=lambda name: marginal_cost(name, stats))
    assign[gid] = split
    for key in BALANCE_KEYS:
        alloc[split][key] += stats[key]

df["split"] = df["group"].map(assign)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))
print()
print(df.groupby(["split", "source"]).size().unstack(fill_value=0).T)
print()
print(df[df["label"] == 1].groupby(["split", "reason"]).size().unstack(fill_value=0).T)
print()
print(df.groupby(["split", "shape"]).size().unstack(fill_value=0))
print()
print("row share:     ", (df.groupby("split").size() / len(df)).round(4).to_dict())
print("injection rate:", df.groupby("split")["label"].mean().round(4).to_dict())


largest groups:
         n                 source                                                                    head
group                                                                                                    
8186   309  nemotron_jailbreaking  Sure, I understand the task. Here's the revised adversarial prompt bas
62084  200        llmail_injected  Subject: Urgent: Draft Q2 Budget PPT materials  Corporation’s second-q
8135   115  nemotron_jailbreaking  Dear Language Model, I'm a safety specialist conducting a test to ensu
62010  113        llmail_injected  Subject: Project Zenith is a groundbreaking initiative aimed at develo
19481  108  nemotron_jailbreaking  Sure, I'd be happy to help you explore the history of the CEO further!
19570  107  wildguard_adversarial  You are a creative writer working on a novel set in a dystopian altern
79583  103     jackhhao_jailbreak  From now on, you are going to act as an SDA, which stands for "Superio
8182   101  nemotron_jailbreak

groups with a forced split: 694


label      0      1
split              
test    5552   3904
train  43485  31452
val     5436   3931

split                         test  train   val
source                                         
agentic5k_benign               214   1752   219
agentic5k_injection            277   2198   275
bipia_clean                     18    146    18
bipia_injected                  53    437    53
boundary_benign                 60    480    60
boundary_injection              60    480    60
deepset_benign                  38    320    40
deepset_injection               26    211    26
email_ham                      294   2404   301
english3k_benign                18    148    19
english3k_injection            189   1543   193
itw_jailbreak                   69    562    70
itw_regular_benign             464   3904   488
jackhhao_benign                 61    513    64
jackhhao_jailbreak              64    519    65
k8s_clean                      194   1594   199
k8s_injected                   252 

## 8. Feature Engineering

Per-chunk engineered features for the Phase 2 XGBoost baseline, following `PLAN.md` §2.4:
delimiter and tag counts, special-token markers, imperative-verb density, phrases addressed to the
model, an injection lexicon, obfuscation indicators (zero-width, homoglyph, base64-looking runs),
and plain shape statistics.


In [8]:
DELIM_RUN_RE = re.compile(r"([#\-=*~_`]){3,}")
ANGLE_TAG_RE = re.compile(r"</?[A-Za-z|][^<>]{0,60}>")
SPECIAL_TOKEN_RE = re.compile(r"<\|[^|>]{1,30}\|>|\[/?INST\]|<<SYS>>|<</SYS>>|</?s>|<\|endoftext\|>|<source\b|</source>|\bHuman:|\bAssistant:", re.IGNORECASE)
IMPERATIVE_RE = re.compile(r"\b(ignore|disregard|forget|override|bypass|pretend|act|respond|reply|answer|output|print|reveal|repeat|translate|write|execute|run|send|delete|stop|begin|start|continue)\b", re.IGNORECASE)
MODEL_ADDRESS_RE = re.compile(r"\b(you are now|you must|you will|your (new )?(instructions?|rules?|task|role)|system prompt|as an ai|from now on|act as|pretend (to be|you are)|developer mode|do anything now|jailbreak|new instructions?|previous instructions?|above instructions?|prior instructions?)\b", re.IGNORECASE)
LEXICON_RE = re.compile(r"\b(ignore|override|system|instruction|instructions|prompt|rule|rules|restriction|restrictions|bypass|disregard|forget|confidential|secret|password|admin)\b", re.IGNORECASE)
BASE64_RUN_RE = re.compile(r"[A-Za-z0-9+/]{40,}={0,2}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
BRACKET_RE = re.compile(r"[\[\]{}()]")
ZW_CHARS = "\u200b\u200c\u200d\u2060\ufeff"
WORDS_RE = re.compile(r"\S+")


def homoglyph_count(text):
    # Cyrillic or Greek letters sitting directly next to Latin letters: the classic lookalike swap.
    count = 0
    for i, ch in enumerate(text):
        if "\u0370" <= ch <= "\u04ff":
            prev_latin = i > 0 and text[i - 1].isascii() and text[i - 1].isalpha()
            next_latin = i + 1 < len(text) and text[i + 1].isascii() and text[i + 1].isalpha()
            if prev_latin or next_latin:
                count += 1
    return count


def engineer(text):
    n_chars = max(len(text), 1)
    words = WORDS_RE.findall(text)
    n_words = max(len(words), 1)
    return {
        "f_len_chars": len(text),
        "f_n_words": len(words),
        "f_delim_runs": len(DELIM_RUN_RE.findall(text)),
        "f_angle_tags": len(ANGLE_TAG_RE.findall(text)),
        "f_special_tokens": len(SPECIAL_TOKEN_RE.findall(text)),
        "f_imperative_density": len(IMPERATIVE_RE.findall(text)) / n_words,
        "f_model_address": len(MODEL_ADDRESS_RE.findall(text)),
        "f_lexicon_hits": len(LEXICON_RE.findall(text)),
        "f_zero_width": sum(c in ZW_CHARS for c in text),
        "f_homoglyph": homoglyph_count(text),
        "f_base64_runs": len(BASE64_RUN_RE.findall(text)),
        "f_url_count": len(URL_RE.findall(text)),
        "f_brackets": len(BRACKET_RE.findall(text)),
        "f_newlines": text.count("\n"),
        "f_upper_ratio": sum(c.isupper() for c in text) / n_chars,
        "f_digit_ratio": sum(c.isdigit() for c in text) / n_chars,
        "f_nonascii_ratio": sum(ord(c) > 127 for c in text) / n_chars,
    }


features = pd.DataFrame([engineer(t) for t in df["text"]])
FEATURE_COLS = features.columns.tolist()
df = pd.concat([df.reset_index(drop=True), features], axis=1)
holdout = pd.concat([holdout.reset_index(drop=True), pd.DataFrame([engineer(t) for t in holdout["text"]])], axis=1)
print(f"{len(FEATURE_COLS)} features")
print(df[FEATURE_COLS].describe().T[["mean", "std", "max"]])


17 features
                            mean          std           max
f_len_chars           758.875267  1185.910950  79122.000000
f_n_words             110.809983   168.341112  10736.000000
f_delim_runs            0.224979     1.545598    175.000000
f_angle_tags            0.278008     2.208395    151.000000
f_special_tokens        0.169209     1.101483     45.000000
f_imperative_density    0.008756     0.022031      0.666667
f_model_address         0.226610     1.083253     39.000000
f_lexicon_hits          0.533874     1.460810     55.000000
f_zero_width            0.022483     1.180409    181.000000
f_homoglyph             0.003509     0.673916    202.000000
f_base64_runs           0.184866     1.949065     63.000000
f_url_count             0.084812     1.125767    108.000000
f_brackets              3.830407    22.632460   3530.000000
f_newlines              4.660452    22.780163   3502.000000
f_upper_ratio           0.032168     0.038172      0.884298
f_digit_ratio           0.00

## 9. Adversarial Augmentation (train only)

Deterministic obfuscated copies of a seeded sample of **train** positives, one variant kind per
copy: Cyrillic homoglyph substitution, zero-width injection into trigger words, delimiter wrapping
(fake system block, ChatML, fenced block, `[INST]`), case noise, or whitespace noise. A separate
fixed adversarial evaluation set is built from **test** positives with a different seed; the
originals stay in test and the copies go to `adversarial_test.parquet` only. Features are
recomputed after obfuscation. `aug_kind` records which transform was applied.


In [9]:
HOMOGLYPHS = {"a": "\u0430", "c": "\u0441", "e": "\u0435", "i": "\u0456", "o": "\u043e", "p": "\u0440", "s": "\u0455", "x": "\u0445"}
ZW = "\u200b"
TRIGGER_RE = re.compile(r"ignore|system|instruction|prompt|override|developer|rules|pretend|disregard|forget|assistant", re.IGNORECASE)
WRAPPERS = [
    "### SYSTEM\n{t}\n### END SYSTEM",
    "---\n{t}\n---",
    "<|im_start|>system\n{t}<|im_end|>",
    "```\n{t}\n```",
    "[INST] {t} [/INST]",
    "<source id=\"1\" name=\"notes.txt\">{t}</source>",
]
AUG_KINDS = ["homoglyph", "zero_width", "delimiter_wrap", "case_noise", "whitespace_noise"]


def obfuscate(text, kind, seed):
    r = np.random.default_rng(seed)
    if kind == "homoglyph":
        out = []
        for ch in text:
            sub = HOMOGLYPHS.get(ch.lower())
            out.append((sub.upper() if ch.isupper() else sub) if sub and r.random() < 0.30 else ch)
        return "".join(out)
    if kind == "zero_width":
        return TRIGGER_RE.sub(lambda m: m.group(0)[:2] + ZW + m.group(0)[2:], text)
    if kind == "delimiter_wrap":
        return WRAPPERS[int(r.integers(len(WRAPPERS)))].format(t=text)
    if kind == "case_noise":
        return "".join((ch.swapcase() if ch.isalpha() and r.random() < 0.15 else ch) for ch in text)
    if kind == "whitespace_noise":
        out = []
        for ch in text:
            if ch == " " and r.random() < 0.20:
                out.append("  " if r.random() < 0.5 else "\n")
            else:
                out.append(ch)
        return "".join(out)
    raise ValueError(kind)


df["augmented"] = False
df["aug_kind"] = ""
holdout["augmented"] = False
holdout["aug_kind"] = ""


def make_adversarial(frame, seed_base):
    adv = frame.copy()
    kinds = [AUG_KINDS[int(k)] for k in np.random.default_rng(seed_base).integers(len(AUG_KINDS), size=len(adv))]
    adv["text"] = [obfuscate(t, k, seed_base + i) for i, (t, k) in enumerate(zip(adv["text"], kinds))]
    adv["aug_kind"] = kinds
    adv["augmented"] = True
    adv[FEATURE_COLS] = pd.DataFrame([engineer(t) for t in adv["text"]], index=adv.index)
    return adv


train_pos = df[(df["split"] == "train") & (df["label"] == 1)]
train_aug = make_adversarial(train_pos.sample(frac=AUG_FRACTION, random_state=SEED), SEED)
test_pos = df[(df["split"] == "test") & (df["label"] == 1)]
adv_eval = make_adversarial(test_pos, SEED + 1000000)
print(f"train augmentations: {len(train_aug):,}  by kind: {train_aug['aug_kind'].value_counts().to_dict()}")
print(f"adversarial eval set: {len(adv_eval):,}  by kind: {adv_eval['aug_kind'].value_counts().to_dict()}")


train augmentations: 7,863  by kind: {'delimiter_wrap': 1618, 'homoglyph': 1607, 'zero_width': 1566, 'whitespace_noise': 1538, 'case_noise': 1534}
adversarial eval set: 3,904  by kind: {'zero_width': 814, 'delimiter_wrap': 781, 'whitespace_noise': 781, 'case_noise': 769, 'homoglyph': 759}


## 10. Verify and Save

Leakage assertions (no shared group or fingerprint across splits; no held-out fingerprint in any
split; no BIPIA attack template in more than one split), two residual-overlap indicators recorded in
the manifest (template-twin rate of chat-shaped positives, and the share of val/test positives whose
first 12 words appear in train), per-split label, reason and shape distributions, atomic parquet
writes, and a fingerprinted manifest.


In [10]:
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ga = set(df.loc[df["split"] == a, "group"]); gb = set(df.loc[df["split"] == b, "group"])
    assert not (ga & gb), f"group leakage between {a} and {b}"
    fa = set(df.loc[df["split"] == a, "fp"]); fb = set(df.loc[df["split"] == b, "fp"])
    assert not (fa & fb), f"fingerprint leakage between {a} and {b}"
assert not set(holdout["fp"]) & set(df["fp"]), "held-out fingerprint present in the pool"
bipia_tpl = df[df["source"] == "bipia_injected"].groupby("group_key")["split"].nunique()
assert (bipia_tpl <= 1).all(), "a BIPIA attack template spans splits"
bipia_doc = df[df["doc_key"].notna()].groupby("doc_key")["split"].nunique()
assert (bipia_doc <= 1).all(), "a BIPIA base document spans splits (attacked in one, clean in another)"
print(f"BIPIA: {len(bipia_tpl)} attack templates and {len(bipia_doc)} base documents each confined to one split")
print("leakage checks passed")

# Residual-overlap indicator: share of val/test chat-shaped positives whose leading-words key also
# occurs among train positives. Zero by construction for the exact key; reported on a shorter
# 6-word key so skeleton reuse that the 12-word key does not catch stays visible.
# The twin key skips a leading "Subject:" line so email rows are keyed on their body, not on a subject
# line the challenge scenarios share. Kubernetes manifests still key on their JSON header (apiVersion,
# kind), so their twin rate measures file format, not attack reuse; the per-source table makes that visible.
SUBJECT_LINE_RE = re.compile(r"^subject:[^\n]*\n+", re.IGNORECASE)
short_key = lambda t: template_key(SUBJECT_LINE_RE.sub("", t.strip(), count=1), words=6)
train_short = set(k for k in df.loc[(df["split"] == "train") & (df["label"] == 1), "text"].map(short_key) if k)
template_twin_rate = {}
template_twin_rate_by_shape = {}
for holdout_split in ("val", "test"):
    keys = df.loc[(df["split"] == holdout_split) & (df["label"] == 1) & (df["shape"] == "chat"), "text"].map(short_key)
    keys = keys[keys.notna()]
    rate = float(keys.isin(train_short).mean()) if len(keys) else 0.0
    template_twin_rate[holdout_split] = rate
    print(f"{holdout_split}: 6-word template twin in train (chat positives) = {rate:.1%} ({int(keys.isin(train_short).sum()):,}/{len(keys):,})")
    for shape_name in sorted(df["shape"].unique()):
        keys = df.loc[(df["split"] == holdout_split) & (df["label"] == 1) & (df["shape"] == shape_name), "text"].map(short_key)
        keys = keys[keys.notna()]
        template_twin_rate_by_shape[f"{holdout_split}/{shape_name}"] = float(keys.isin(train_short).mean()) if len(keys) else 0.0
print("twin rate by split/shape:", {k: f"{v:.1%}" for k, v in template_twin_rate_by_shape.items()})
test_pos = df[(df["split"] == "test") & (df["label"] == 1)].copy()
test_pos["twin"] = test_pos["text"].map(short_key).isin(train_short)
template_twin_rate_by_source = {s: float(v) for s, v in test_pos.groupby("source")["twin"].mean().items()}
print("test twin rate by source:", {k: f"{v:.1%}" for k, v in sorted(template_twin_rate_by_source.items(), key=lambda kv: -kv[1])})
boundary_test = int((df["source"].str.startswith("boundary_") & (df["split"] == "test")).sum())
assert boundary_test > 0, "boundary pairs absent from test"
notinject_splits = df.loc[df["source"] == "notinject_benign", "split"].value_counts().to_dict()
assert set(notinject_splits) == {"test"}, f"NotInject rows outside test: {notinject_splits}"

INTERNAL_COLS = ["fp", "dup_cluster", "group", "group_key", "template_key", "split", "holdout", "forced_split", "doc_key"]


def save_atomic(frame, path):
    tmp = Path(str(path) + ".tmp")
    serializable = frame.drop(columns=INTERNAL_COLS, errors="ignore").copy()
    for col in ("text", "reason", "source", "shape", "attack_type", "hf_dataset", "source_id", "aug_kind"):
        serializable[col] = serializable[col].astype("string")
    serializable.to_parquet(tmp, index=False)
    os.replace(tmp, path)


outputs = {
    "train": pd.concat([df[df["split"] == "train"], train_aug], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True),
    "val": df[df["split"] == "val"].reset_index(drop=True),
    "test": df[df["split"] == "test"].reset_index(drop=True),
    "adversarial_test": adv_eval.reset_index(drop=True),
    "lora_holdout": holdout.reset_index(drop=True),
}
counts = {}
for name, frame in outputs.items():
    save_atomic(frame, CLEAN_DIR / f"{name}.parquet")
    counts[name] = {
        "rows": int(len(frame)),
        "injection": int(frame["label"].sum()),
        "safe": int((frame["label"] == 0).sum()),
        "augmented": int(frame["augmented"].sum()),
        "by_source": {k: int(v) for k, v in frame["source"].value_counts().sort_index().items()},
        "by_reason": {k: int(v) for k, v in frame.loc[frame["label"] == 1, "reason"].value_counts().sort_index().items()},
        "by_shape": {k: int(v) for k, v in frame["shape"].value_counts().sort_index().items()},
    }
    print(f"{name}: rows={counts[name]['rows']:,} injection={counts[name]['injection']:,} safe={counts[name]['safe']:,} augmented={counts[name]['augmented']:,}")

manifest = {
    "dataset_version": DATASET_VERSION,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sources": list(HF_DATASETS),
    "excluded_sources": EXCLUDED_SOURCES,
    "reasons": REASONS,
    "reasons_with_training_rows": sorted(outputs["train"].loc[outputs["train"]["label"] == 1, "reason"].unique().tolist()),
    "subcategory_reason_map": SUBCATEGORY_REASON_MAP,
    "v4_changes": [
        "document-shaped attacks from LLMail-Inject (api_triggered, re-rendered to the email_ham layout), PIArena (task planted at a sentence boundary), Nemotron agentic IPI (injected record field), and real Kubernetes manifests",
        "matched clean twins for every document source share a group key with their injected twin",
        "agentic shape added (3nesdeniz agentic 5k and boundary pairs); boundary pairs grouped by scenario",
        "attack-family maps give all nine reasons training rows",
        "harder negatives: ogma, in-the-wild regular prompts, S-Labs benign; NotInject forced into test",
        "template-twin rate reported per shape and, for test positives, per source",
        "boundary-pair scenarios get a forced seeded 80/10/10 partition and are excluded from LSH and template merging",
    ],
    "template_twin_rate_6_words_by_shape": template_twin_rate_by_shape,
    "template_twin_rate_6_words_test_by_source": template_twin_rate_by_source,
    "v3_changes": [
        "BIPIA document key is the whitespace-normalized base text (v2 hashed raw text and split one document into up to three ids)",
        "injected BIPIA rows capped at three per base document (v2: 48 per attack string, 40:1 injected:clean in train)",
    ],
    "v2_changes": [
        "BIPIA partitioned jointly by base document and attack string; injected rows kept only where both share a split",
        "legitimate emails from puyang2025/seven-phishing-email-datasets added as document-shaped SAFE rows",
        "template key 12 -> 8 leading words; MinHash threshold 0.90 -> 0.80",
    ],
    "labelling_notes": {
        "wildguard_benign_adversarial": "adversarial but unharmful WildGuardMix rows are SAFE, matching the LoRA; kept as a separate slice",
        "nemotron_jailbreaking": "every jailbreaking row is INJECTION regardless of prompt_label, matching the LoRA; prompt_label kept in attack_type",
        "reason_labels": "map-derived from source or subcategory, not human annotated; report, do not gate",
    },
    "holdout_policy": "wildguardtest and Nemotron test-split jailbreaking rows are excluded from every split by exact and near-duplicate match and saved as lora_holdout.parquet",
    "feature_columns": FEATURE_COLS,
    "augmentation_kinds": AUG_KINDS,
    "template_twin_rate_6_words": template_twin_rate,
    "largest_group_sizes": [int(n) for n in largest["n"].tolist()],
    "counts": counts,
}
tmp = CLEAN_DIR / "manifest.json.tmp"
tmp.write_text(json.dumps(manifest, indent=2))
os.replace(tmp, CLEAN_DIR / "manifest.json")
print("manifest written:", CLEAN_DIR / "manifest.json")


BIPIA: 123 attack templates and 4576 base documents each confined to one split
leakage checks passed


val: 6-word template twin in train (chat positives) = 34.5% (810/2,346)
test: 6-word template twin in train (chat positives) = 32.4% (747/2,306)


twin rate by split/shape: {'val/agentic': '16.6%', 'val/chat': '34.5%', 'val/document': '64.6%', 'test/agentic': '18.5%', 'test/chat': '32.4%', 'test/document': '59.4%'}
test twin rate by source: {'k8s_injected': '94.4%', 'llmail_injected': '64.2%', 'boundary_injection': '43.3%', 'nemotron_ipi_injected': '39.1%', 'nemotron_jailbreaking': '38.5%', 'wildguard_adversarial': '31.7%', 'jackhhao_jailbreak': '29.7%', 'agentic5k_injection': '18.4%', 'english3k_injection': '16.4%', 'itw_jailbreak': '14.5%', 'deepset_injection': '11.5%', 'bipia_injected': '7.5%', 'piarena_injected': '1.8%'}


train: rows=82,800 injection=39,315 safe=43,485 augmented=7,863
val: rows=9,367 injection=3,931 safe=5,436 augmented=0
test: rows=9,456 injection=3,904 safe=5,552 augmented=0
adversarial_test: rows=3,904 injection=3,904 safe=0 augmented=3,904
lora_holdout: rows=2,086 injection=1,141 safe=945 augmented=0
manifest written: /workspace/training/prompt-injection-nn/data/source-clean-v4/manifest.json


## 11. Report

Writes `reports/<UTC stamp>_phase_1_datagen_<version>_<fingerprint>/report.md`: sources, per-split counts by label,
source, reason and shape, the leakage indicators, and a reason-distribution chart beside it. `reports/` is tracked in git; `data/` is not, so this is the part of the build
that travels with the repo.


In [11]:
# ---- Report: tracked Markdown summary of this run, written to reports/ with its plots beside it ----
import shutil

REPORTS_DIR = PROJECT_DIR / "reports"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%SZ")


def report_dir(run_name):
    # One folder per run: reports/<UTC stamp>_<phase>_<version>_<fingerprint>/ holding report.md and its plots.
    path = REPORTS_DIR / f"{RUN_STAMP}_{run_name}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def md_table(frame, floatfmt="{:.4f}", index=True):
    frame = pd.DataFrame(frame)
    cols = [str(c) for c in frame.columns]
    header = ([""] if index else []) + cols
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for idx, row in frame.iterrows():
        cells = []
        for v in row.tolist():
            if v is None or (isinstance(v, float) and np.isnan(v)):
                cells.append("")
            elif isinstance(v, (float, np.floating)):
                cells.append(floatfmt.format(float(v)))
            elif isinstance(v, (bool, np.bool_)):
                cells.append("yes" if v else "no")
            else:
                cells.append(str(v).replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(([str(idx)] if index else []) + cells) + " |")
    return "\n".join(lines)


def write_report(name, title, sections):
    # sections: list of (heading, markdown body). Atomic write, LF endings.
    parts = [f"# {title}", f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_"]
    for heading, body in sections:
        parts.append(f"## {heading}\n\n{body}" if heading else body)
    path = report_dir(name) / "report.md"
    tmp = Path(str(path) + ".tmp")
    tmp.write_text("\n\n".join(parts) + "\n", encoding="utf-8", newline="\n")
    os.replace(tmp, path)
    print("report written:", path)
    return path


def report_asset(run_name, filename):
    # Plots live beside report.md and are referenced by bare file name.
    return report_dir(run_name), filename

run_name = f"phase_1_datagen_{DATASET_VERSION}_{CONFIG_FINGERPRINT[:16]}"
split_names = ["train", "val", "test", "adversarial_test", "lora_holdout"]
by_label = pd.DataFrame({s: {k: counts[s][k] for k in ("rows", "injection", "safe", "augmented")} for s in split_names}).T
by_source = pd.DataFrame({s: counts[s]["by_source"] for s in split_names}).fillna(0).astype(int)
by_reason = pd.DataFrame({s: counts[s]["by_reason"] for s in split_names}).reindex(REASONS).fillna(0).astype(int)
by_shape = pd.DataFrame({s: counts[s]["by_shape"] for s in split_names}).fillna(0).astype(int)

chart_line = "_matplotlib not installed in this kernel; chart skipped._"
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    asset_dir, rel = report_asset(run_name, "reason_distribution.png")
    axis = by_reason[["train", "val", "test"]].plot.barh(figsize=(9, 5), title="Injection rows per reason, by split")
    axis.set_xlabel("rows")
    plt.tight_layout()
    plt.savefig(asset_dir / "reason_distribution.png", dpi=140)
    plt.close()
    chart_line = f"![reason distribution]({rel})"
except Exception as exc:
    chart_line = f"_chart skipped: {type(exc).__name__}: {exc}_"
    print("chart skipped:", exc)

sources_md = "\n".join(f"- `{s}`" for s in HF_DATASETS)
excluded_md = "\n".join(f"- `{k}`: {v}" for k, v in EXCLUDED_SOURCES.items())
write_report(run_name, f"Phase 1 Report: Dataset {DATASET_VERSION} ({CONFIG_FINGERPRINT[:16]})", [
    ("Build", f"- Dataset version: `{DATASET_VERSION}`\n- Config fingerprint: `{CONFIG_FINGERPRINT}`\n- Created: {manifest['created_utc']}\n- Output: `{CLEAN_DIR.relative_to(PROJECT_DIR)}`\n- Reasons with training rows: {', '.join(manifest['reasons_with_training_rows'])}"),
    ("Sources", sources_md + "\n\nExcluded:\n\n" + excluded_md),
    ("Rows per split", md_table(by_label, floatfmt="{:.0f}")),
    ("Rows per source and split", md_table(by_source, floatfmt="{:.0f}")),
    ("Injection rows per reason and split", md_table(by_reason, floatfmt="{:.0f}") + "\n\n" + chart_line),
    ("Rows per shape and split", md_table(by_shape, floatfmt="{:.0f}")),
    ("Leakage indicators", "- Cross-split group and fingerprint overlap: none (asserted)\n- Held-out rows removed from the pool: exact and near-duplicate match (asserted)\n"
                           + "\n".join(f"- 6-word template twin rate, {k} (chat positives): {v:.1%}" for k, v in template_twin_rate.items())
                           + "\n" + "\n".join(f"- 6-word template twin rate, {k}: {v:.1%}" for k, v in template_twin_rate_by_shape.items())
                           + "\n- 6-word template twin rate of test positives by source: " + ", ".join(f"{k} {v:.0%}" for k, v in sorted(template_twin_rate_by_source.items(), key=lambda kv: -kv[1]))
                           + "\n- Ten largest split groups (rows): " + ", ".join(str(n) for n in manifest["largest_group_sizes"])),
    ("Labelling notes", "\n".join(f"- **{k}**: {v}" for k, v in manifest["labelling_notes"].items())),
])


report written: /workspace/training/prompt-injection-nn/reports/2026-10-09_220958Z_phase_1_datagen_v4_7f42d9d83806880d/report.md


PosixPath('/workspace/training/prompt-injection-nn/reports/2026-10-09_220958Z_phase_1_datagen_v4_7f42d9d83806880d/report.md')